In [19]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import yaml
import spacy
from corextopic import corextopic as ct
from dvclive import Live
from matplotlib.figure import Figure
from spacy.tokens import DocBin

from job_post_nlp.utils.interactive import try_inter

try_inter()
from job_post_nlp.prepare import corpus_unpack, register_extensions, load_data,register_extensions, load_texts # noqa: E402
from job_post_nlp.utils.find_project_root import find_project_root  # noqa: E402
from job_post_nlp.evaluate import load_model  # noqa: E402
from job_post_nlp.train import load_corpus_split, load_tdm  # noqa: E402
import helpfuncs as hf

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
# Process
data = hf.load_everything()
tdm = data['tdm']
tdm_info = data['tdm_info']


In [21]:
anchors = hf.read_anchors_from_yaml('note.yaml')

In [22]:
anchors

[['fleksibel',
  'fleksibilitet',
  'fleksibel arbejdstid',
  'fleksibel arbejdstid',
  'flekstid',
  'flekstidsordning',
  'flextid',
  'flextidsordning',
  'fixtid',
  'fikstid'],
 ['familievenlig', 'familievenlige', 'familieliv'],
 ['hjemmearbejde', 'hjemmearbejdsplads', 'hjemmearbejdsdag', 'hjemmefra'],
 ['vagtskema', 'vagtplan', 'spidsbelastning'],
 ['natarbejde',
  'weekend',
  'weekendarbejde',
  'helligdage',
  'nattevagt',
  'nattevagte',
  'nattevagterne',
  'aften'],
 ['vagtplan'],
 ['ambitiøs',
  'ambition',
  'udvikling',
  'karriere',
  'udviklingsmuligheder',
  'udviklingsplaner',
  'udviklingsforløb',
  'udviklingsprogram'],
 ['faglig', 'faglighed', 'sparring'],
 ['stillingen',
  'samtaler',
  'snarest muligt',
  'mulig',
  'ansøgning',
  'ringe',
  'ansøgning mail',
  'ansøgning',
  'sende',
  'cv',
  'sende ansøgning',
  'ansøgning cv',
  'send',
  'send ansøgning',
  'ansøgning sende',
  'modtage ansøgning'],
 ['køn',
  'etnisk',
  'religion',
  'religion etnisk',
  

In [23]:
# Print most common tokens in tdm
freq = tdm.sum(axis=0).A1
# Sort tokens by frequency
freq_sort = np.argsort(freq)[::-1]
freq_sorted = freq[freq_sort]

In [24]:
sorted_tokens = np.array(tdm_info['vocab'] )[freq_sort]
sorted_tokens

array(['søge', 'arbejde', 'god', ..., 'aabogade', 'aabenraaa',
       'aabenraaby'], shape=(568817,), dtype='<U100')

In [25]:
topn = 1000
print(f"Top {topn} tokens by frequency:")
for token, frequency in zip(sorted_tokens[:topn], freq_sorted[:topn]):
    print(f"{token}: {frequency}")

Top 1000 tokens by frequency:
søge: 1531923
arbejde: 1490004
god: 1171709
stilling: 1169962
erfaring: 1008833
samt: 1007670
samarbejde: 938583
tilbyde: 930904
opgave: 903639
ansøgning: 880115
del: 875645
stor: 866371
mulighed: 862761
forvente: 814158
medarbejder: 808202
udvikling: 791042
både: 761137
faglig: 751767
time: 741720
job: 734931
gerne: 662382
ny: 650198
ikke: 647745
uge: 633926
sende: 624324
høj: 622681
hverdag: 621247
få: 620518
kollega: 614649
team: 609795


se: 596817
fokus: 594253
løn: 590805
overenskomst: 576767
lyst: 572310
spændende: 564698
år: 561468
inden: 558100
skabe: 542838
tage: 533331
indgå: 518454
ansvar: 514498
tæt: 512861
ønske: 512266
selvstændig: 506807
bestå: 504584
udvikle: 503000
hos: 499476
arbejdsplads: 484457
kontakte: 483427
dag: 477031
gælde: 472868
din: 470819
kollegaa: 463079
fleksibel: 462528
uddannelse: 460475
løbende: 446357
velkommen: 443992
cv: 435850
afdeling: 433514
dygtig: 432842
forskellig: 431699
ansættelse: 426637
kommune: 420597
relevant: 419801
vigtig: 417658
tale: 416958
arbejdstid: 409557
dansk: 406560
område: 405950
arbejdsmiljø: 401910
fast: 399228
tid: 394862
ansætte: 394728
sætte: 392977
positiv: 388185
gælde overenskomst: 385547
afholde: 384880
oplysning: 382446
personlig: 379014
mulig: 377706
hel: 376974
daglig: 375333
ansættelsesvilkår: 374929
sikre: 371935
virksomhed: 368757
læse: 366624
ansøgningsfrist: 364151
forhold: 360498
give: 360023
kontakt: 356429
indenfor: 355952
spørgsmål: 350565

In [26]:
topn = 1000
print(f"Bottom {topn} tokens by frequency:")
for token, frequency in zip(sorted_tokens[-topn:], freq_sorted[-topn:]):
    print(f"{token}: {frequency}")

Bottom 1000 tokens by frequency:
skoledagén: 2
skoledagtilbudsrådgiver: 2
skoledagslængde: 2
skoledagsbehandlingsområde: 2
skoledagsarrangement: 2
skolerepræsentanter: 2
skolerepræsentant: 2
acelerere: 2
trusselsvurderingsenhed: 2
trusselsvurderinger: 2
moonsena: 2
moonis: 2
moonflower: 2
mooncare: 2
moonboon: 2
moohko: 2
truckjob: 2
truckføring: 2
truckførerne: 2
truckførerkort: 2
truckførerhasselager: 2
monstro: 2
trustgate: 2
trusseslvurderingsenhed: 2
hospic: 2
hospersonalet: 2
hoslede: 2
hosklinikchefsekretær: 2
hosie: 2
truckkrot: 2
truckkorte: 2
truckkapacitet: 2
accompaniment: 2
landdomænets: 2
landdomæneafdelingen: 2
landdomæneafdeling: 2
landdistriktsvid: 2
landdistriktsudviklings: 2
landdistriktstilskud: 2
hospiceafsnit: 2
øvelseselemente: 2
øvelsesdukker: 2
skolepædagogmedhjælper: 2
skolepædagogjob: 2
skolepå: 2
skolefeltet: 2
skolefamilierådgiver: 2
skolefagsundervisning: 2
acció: 2
accinitet: 2
accidentielt: 2
accesudvikler: 2
accesspunkt: 2
accessoriesområd: 2
accessorie


skolefagstof: 2
øvelsesdel: 2
øvelsescenteret: 2
øvelsesbrug: 2
accessoirers: 2
accessments: 2
lampekonsulenten: 2
lampekass: 2
holtebo: 2
holtbjergsskole: 2
holtbjergskolens: 2
holtbjergfritidscenter: 2
holtab: 2
holstrbro: 2
holstenske: 2
holsten: 2
acceptbrev: 2
acceptabl: 2
accep: 2
montæring: 2
montære: 2
holtets: 2
holteranalyse: 2
holteeller: 2
accessionering: 2
accessed: 2
accepts: 2
accepting: 2
accepterrer: 2
accepteret: 2
accepterere: 2
acceptence: 2
tankbygning: 2
tankblanding: 2
tankafdeling: 2
tanjung: 2
lampebranch: 2
lamotek: 2
lammeskind: 2
accessionsteam: 2
landdistriktsomådet: 2
landdistriktsnetværk: 2
landdistriktsministeriet: 2
landdistriktskoordinatoren: 2
landdistriktskonference: 2
landdistriktshjemmeside: 2
landdistriktsgrupper: 2
tandsbjergs: 2
skolepædagoguddannelse: 2
skolepædagogpædagog: 2
accelererer: 2
acceleratorteknikker: 2
acceleratornedbrud: 2
lanceringsdatoe: 2
lanbrugssmed: 2
landdistriktsordninge: 2
acceleromterdata: 2
acceleromet: 2
skoleforbere: 

In [27]:
freq_sorted[(sorted_tokens =='god overblik')]

array([53165])

In [28]:
freq.shape

(568817,)

In [29]:
(freq==1).sum(), (freq==2).sum(), (freq==3).sum(), (freq==4).sum()

(np.int64(0), np.int64(129021), np.int64(69705), np.int64(45201))

In [30]:
hf.check_anchors(anchors,data)


Checking anchor: fleksibel
'fleksibel' occurs 462528 times in the TDM.
'fleksibilitet' occurs 104468 times in the TDM.
'fleksibel arbejdstid' occurs 32134 times in the TDM.
'fleksibel arbejdstid' occurs 32134 times in the TDM.
'flekstid' occurs 11214 times in the TDM.
'flekstidsordning' occurs 5758 times in the TDM.
'flextid' occurs 6775 times in the TDM.
'flextidsordning' occurs 1328 times in the TDM.
'fixtid' occurs 124 times in the TDM.
'fikstid' occurs 22 times in the TDM.

Checking anchor: familievenlig
'familievenlig' occurs 4954 times in the TDM.
'familievenlige' occurs 101 times in the TDM.
'familieliv' occurs 6870 times in the TDM.

Checking anchor: hjemmearbejde
'hjemmearbejde' occurs 5195 times in the TDM.
'hjemmearbejdsplads' occurs 2437 times in the TDM.
'hjemmearbejdsdag' occurs 4105 times in the TDM.
'hjemmefra' occurs 9532 times in the TDM.

Checking anchor: vagtskema
'vagtskema' occurs 1581 times in the TDM.
'vagtplan' occurs 38305 times in the TDM.
'spidsbelastning' 

In [13]:
hf.find_similar_words('fleksib',data)

Similar words to 'fleksib':
'fleksibel' occurs 462528 times in the TDM.
'fleksibilitet' occurs 104468 times in the TDM.
'fleksibel arbejdstid' occurs 32134 times in the TDM.
'fleksibel forhold' occurs 28457 times in the TDM.
'mødestabil fleksibel' occurs 19839 times in the TDM.
'fleksibelt' occurs 19622 times in the TDM.
'fleksible' occurs 13850 times in the TDM.
'fleksibilit' occurs 13266 times in the TDM.
'fleksibl' occurs 1619 times in the TDM.
'fleksib' occurs 194 times in the TDM.
'vagtfleksibilitet' occurs 143 times in the TDM.
'fleksibil' occurs 136 times in the TDM.
'fleksibilitetstillæg' occurs 127 times in the TDM.
'fleksibe' occurs 102 times in the TDM.
'fleksibiltet' occurs 64 times in the TDM.
'erfleksibel' occurs 55 times in the TDM.
'fleksibelog' occurs 53 times in the TDM.
'fleksibilite' occurs 50 times in the TDM.
'ufleksibel' occurs 49 times in the TDM.
'fleksiblehold' occurs 39 times in the TDM.
'ogfleksibel' occurs 34 times in the TDM.
'fleksibitet' occurs 30 times 

In [14]:
hf.find_similar_words('flekstid',data)

Similar words to 'flekstid':
'flekstid' occurs 11214 times in the TDM.
'flekstidsordning' occurs 5758 times in the TDM.
'flekstidsaftale' occurs 187 times in the TDM.
'flekstidsaftal' occurs 118 times in the TDM.
'flekstidssystem' occurs 22 times in the TDM.
'flekstidordning' occurs 8 times in the TDM.
'flekstider' occurs 7 times in the TDM.
'flekstidsadministratione' occurs 6 times in the TDM.
'flekstidsordninger' occurs 6 times in the TDM.
'flekstidsregler' occurs 6 times in the TDM.
'flekstidsordni' occurs 4 times in the TDM.
'harflekstidsordning' occurs 4 times in the TDM.
'flekstidsadministration' occurs 3 times in the TDM.
'flekstidsansatte' occurs 3 times in the TDM.
'flekstidso' occurs 3 times in the TDM.
'flekstidsskema' occurs 3 times in the TDM.
'harflekstid' occurs 3 times in the TDM.
'erflekstid' occurs 2 times in the TDM.
'flekstiden' occurs 2 times in the TDM.
'flekstidsløsning' occurs 2 times in the TDM.
'flekstidsopfølgning' occurs 2 times in the TDM.
'flekstidsregnska

In [15]:
hf.find_similar_words('hjemmefra',data)

Similar words to 'hjemmefra':
'hjemmefra' occurs 9532 times in the TDM.
'hjemmefradu' occurs 37 times in the TDM.
'akuthjemmefra' occurs 2 times in the TDM.


In [16]:
hf.find_similar_words('ministeri',data)

Similar words to 'ministeri':
'finansministeri' occurs 67003 times in the TDM.
'overenskomst finansministeri' occurs 29408 times in the TDM.
'forsvarsministeri' occurs 20189 times in the TDM.
'ministeri' occurs 10956 times in the TDM.
'ministerium' occurs 9718 times in the TDM.
'kirkeministerium' occurs 7001 times in the TDM.
'forsvarsministeriets' occurs 6390 times in the TDM.
'skatteministeriet' occurs 6344 times in the TDM.
'skatteministeri' occurs 6018 times in the TDM.
'skatteministeriets' occurs 4244 times in the TDM.
'undervisningsministeriets' occurs 2964 times in the TDM.
'fødevareministeri' occurs 2730 times in the TDM.
'udenrigsministeriet' occurs 2394 times in the TDM.
'forskningsministeri' occurs 2328 times in the TDM.
'kulturministerium' occurs 1840 times in the TDM.
'boligministeriet' occurs 1543 times in the TDM.
'undervisningsministeriet' occurs 1524 times in the TDM.
'miljøministerium' occurs 1442 times in the TDM.
'udenrigsministeriets' occurs 1358 times in the TDM.
